# Change in active wildfires across Australian states/territories during the last 5 days

## Accessing Wildfire Data via API

In [ ]:
# import necessary libraries
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt


In [88]:
# 1.
# access api url

## satellite: MODIS NRT -> MODIS has a better distribution of acquisition times, leaving less gaps in the final map, and Near Real Time for analysing live data. 
### However, Modis has a worse spatial resolution with 1km instead of 375m like VIIRS, but that is a trade-off I can bear
## area: '112,-44,154,-9' = bounding box coordinates for Australia 
## day range: '5' = data of the last 5 days (FIRMS does not let you load more data with one API request)
## date: None = most recent available data, so today's data

MAP_KEY = '4899a992545cbeb46f9fd0b6a025ef17'
api_url ='https://firms.modaps.eosdis.nasa.gov/api/area/csv/' + MAP_KEY + '/MODIS_NRT/112,-44,154,-9/5' 

# 2.
# read in the data from URL

df_fires = pd.read_csv(api_url)

## does data need to be converted into a dictionary from JSON

# 3.
# have a first glimpse at the data

df_fires.head(5)
#df_fires.shape
#df_fires["acq_date"].unique()

,latitude,longitude,brightness,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_t31,frp,daynight
0,-18.75164,121.89628,311.55,3.45,1.75,2026-05-08,109,Terra,MODIS,16,6.1NRT,295.98,38.06,D
1,-17.50352,122.23376,317.26,3.53,1.77,2026-05-08,109,Terra,MODIS,40,6.1NRT,294.99,68.85,D
2,-17.47719,122.24174,319.67,3.53,1.77,2026-05-08,109,Terra,MODIS,71,6.1NRT,294.76,85.11,D
3,-22.73166,119.79930,307.52,2.69,1.57,2026-05-08,111,Terra,MODIS,61,6.1NRT,293.72,21.54,D
4,-22.72930,119.77369,312.30,2.68,1.57,2026-05-08,111,Terra,MODIS,70,6.1NRT,293.72,35.74,D


## Cleaning and Rearranging Data

### Adding Datetime Column with active time 

In [67]:
# 1. 
# combine the acq_date and acq_time column to one acq_datetime column and set it to an active time format with pandas function to_datetime

## acq_date is a string in the format YYYY-MM_DD, 
## while acq_time is an integer in Greenwich Mean Time (e.g. 603 meaning 6:03), 
## so it needs to be converted to string too (with astype(str)),
## fill it up to 4 numbers with zeros, so that all times have the same length (with str.zfill(4), e.g. 603 -> 0603)
## and save it as the format '%Y-%m-%d %H%M'

df_fires['acq_datetime'] = pd.to_datetime(df_fires['acq_date'] + ' ' + df_fires['acq_time'].astype(str).str.zfill(4), format='%Y-%m-%d %H%M')
df_fires.head()

print (f'Australia GMT timezone datetime value range: {df_fires['acq_datetime'].min()} to {df_fires['acq_datetime'].max()}')

# 2.
# convert GMT into local time?

# 3.
# # Set the timestamp column as the index ?
###hourly_data = hourly_data.set_index("timestamp")

# Notice how 'timestamp' drops down a level to become the index!
###display(hourly_data.head(3))



Australia GMT timezone datetime value range: 2026-05-07 00:30:00 to 2026-05-11 08:33:00


### Converting raw coordinates into geometries

In [78]:
# the projection EPSG:9473 is used for Australia, as it is recommended for national mapping

# convert latitude, longitude values into point geometry and set crs (since no crs extisting yet) with crs="EPSG:9473" to EPSG:9473

gdf_fires = gpd.GeoDataFrame(
    df_fires, geometry=gpd.points_from_xy(df_fires.longitude, df_fires.latitude), crs="EPSG:4326").to_crs(epsg=9473)
print(gdf_fires.crs)
gdf_fires.head()

EPSG:9473


,latitude,longitude,brightness,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_t31,frp,daynight,acq_datetime,geometry
0,-16.10577,131.99382,325.95,3.34,1.72,2026-05-07,30,Terra,MODIS,35,6.1NRT,295.18,106.17,D,2026-05-07 00:30:00,POINT (-664.608 -1713087.591)
1,-16.10273,131.98743,325.90,3.34,1.72,2026-05-07,30,Terra,MODIS,34,6.1NRT,294.90,106.26,D,2026-05-07 00:30:00,POINT (-1351.832 -1712753.001)
2,-16.07522,131.99707,323.35,3.34,1.72,2026-05-07,30,Terra,MODIS,71,6.1NRT,293.82,97.24,D,2026-05-07 00:30:00,POINT (-315.174 -1709724.805)
3,-15.78449,128.34108,310.85,1.72,1.29,2026-05-07,30,Terra,MODIS,64,6.1NRT,294.28,12.40,D,2026-05-07 00:30:00,POINT (-394444.915 -1683387.752)
4,-15.60215,128.25143,312.89,1.68,1.27,2026-05-07,30,Terra,MODIS,68,6.1NRT,295.41,15.11,D,2026-05-07 00:30:00,POINT (-404694.867 -1663630.673)


## Adding a boundary GeoPackage file of States/Territories for spatial analysis

In [ ]:
# 1. Load the shapefile zip of Australian territories and ensure CRS are matching
## Source of the GeoPackage: Australian Bureau of Statistics 
## https://www.abs.gov.au/statistics/standards/australian-statistical-geography-standard-asgs-edition-3/jul2021-jun2026/access-and-downloads/digital-boundary-files
gdf_territories = gpd.read_file(
    "/Users/silviazemp/Desktop/Uni/FS26/Python/project/data/raw/ASGS_2021_Main_Structure_GDA2020.gpkg",
    layer="STE_2021_AUST_GDA2020"
).to_crs(epsg=9473)
 # check for valid geometries as sjoin was not working

# 2. Perform the spatial join
## the strict inner option is chosen, cause fires outside any Australian territories should be dropped (the bounding box includes some parts of Indonesia or Papua New Guinea)
## within is chosen as fires are point data and are either within or outside a polygon, and we only want the ones inside
territories_with_fires = gpd.sjoin(gdf_fires, gdf_territories, how="inner", predicate="within")

# 3. View the joined attribute table
display(territories_with_fires.head(5))
gdf_territories.is_valid.value_counts()
## why no output?

,latitude,longitude,brightness,scan,track,acq_date,acq_time,satellite,instrument,confidence,...,geometry,index_right,STATE_CODE_2021,STATE_NAME_2021,CHANGE_FLAG_2021,CHANGE_LABEL_2021,AUS_CODE_2021,AUS_NAME_2021,AREA_ALBERS_SQKM,ASGS_LOCI_URI_2021
0,-16.10577,131.99382,325.95,3.34,1.72,2026-05-07,30,Terra,MODIS,35,...,POINT (-664.608 -1713087.591),6,7,Northern Territory,0,No change,AUS,Australia,1.348134e+06,http://linked.data.gov.au/dataset/asgsed3/STE/7
1,-16.10273,131.98743,325.90,3.34,1.72,2026-05-07,30,Terra,MODIS,34,...,POINT (-1351.832 -1712753.001),6,7,Northern Territory,0,No change,AUS,Australia,1.348134e+06,http://linked.data.gov.au/dataset/asgsed3/STE/7
2,-16.07522,131.99707,323.35,3.34,1.72,2026-05-07,30,Terra,MODIS,71,...,POINT (-315.174 -1709724.805),6,7,Northern Territory,0,No change,AUS,Australia,1.348134e+06,http://linked.data.gov.au/dataset/asgsed3/STE/7
3,-15.78449,128.34108,310.85,1.72,1.29,2026-05-07,30,Terra,MODIS,64,...,POINT (-394444.915 -1683387.752),4,5,Western Australia,0,No change,AUS,Australia,2.526632e+06,http://linked.data.gov.au/dataset/asgsed3/STE/5
4,-15.60215,128.25143,312.89,1.68,1.27,2026-05-07,30,Terra,MODIS,68,...,POINT (-404694.867 -1663630.673),4,5,Western Australia,0,No change,AUS,Australia,2.526632e+06,http://linked.data.gov.au/dataset/asgsed3/STE/5


True     9
False    1
Name: count, dtype: int64

In [82]:
fire_count = territories_with_fires.groupby("STATE_NAME_2021").size()
display(fire_count)

STATE_NAME_2021
New South Wales        151
Northern Territory     766
Queensland             134
South Australia         16
Tasmania                11
Victoria                27
Western Australia     1176
dtype: int64

## Preparing the Data for a Heatmap

In [89]:
# 1.
# group geometries by time (hourly resolution)
territories_with_fires["time_bin"]= territories_with_fires["acq_datetime"].dt.floor("h") #ist nur nötig bei hourly distribution, sonst identisch mit datetime column

data = []
time_index = []

for time, group in territories_with_fires.groupby("time_bin"):
    
    heat_data = group[["latitude", "longitude"]].values.tolist() # add fire radiative power to show intensity of fires
    
    data.append(heat_data)
    time_index.append(str(time))

# check if it worked
territories_with_fires.sample(5)

,latitude,longitude,brightness,scan,track,acq_date,acq_time,satellite,instrument,confidence,...,index_right,STATE_CODE_2021,STATE_NAME_2021,CHANGE_FLAG_2021,CHANGE_LABEL_2021,AUS_CODE_2021,AUS_NAME_2021,AREA_ALBERS_SQKM,ASGS_LOCI_URI_2021,time_bin
76,-19.81755,123.12191,310.13,1.01,1.01,2026-05-07,33,Terra,MODIS,61,...,4,5,Western Australia,0,No change,AUS,Australia,2.526632e+06,http://linked.data.gov.au/dataset/asgsed3/STE/5,2026-05-07 00:00:00
1339,-32.63583,117.00364,309.21,3.37,1.73,2026-05-09,716,Aqua,MODIS,68,...,4,5,Western Australia,0,No change,AUS,Australia,2.526632e+06,http://linked.data.gov.au/dataset/asgsed3/STE/5,2026-05-09 07:00:00
235,-18.32452,145.44475,310.05,1.22,1.10,2026-05-07,603,Aqua,MODIS,62,...,2,3,Queensland,0,No change,AUS,Australia,1.730171e+06,http://linked.data.gov.au/dataset/asgsed3/STE/3,2026-05-07 06:00:00
743,-17.00190,136.38887,336.96,1.05,1.02,2026-05-07,2331,Terra,MODIS,56,...,6,7,Northern Territory,0,No change,AUS,Australia,1.348134e+06,http://linked.data.gov.au/dataset/asgsed3/STE/7,2026-05-07 23:00:00
863,-31.66773,117.19427,308.15,1.62,1.25,2026-05-08,818,Aqua,MODIS,67,...,4,5,Western Australia,0,No change,AUS,Australia,2.526632e+06,http://linked.data.gov.au/dataset/asgsed3/STE/5,2026-05-08 08:00:00


## Visualising the data with a folium map with State/Territory Polygons and an animated Heatmap of the Fire Distribution

In [ ]:
import folium
from folium.plugins import HeatMapWithTime

# 1. Create basemap for the extent of Australia
aus_map = folium.Map(
    location=[-25.5649, 133.1234], # use the coordinates of Australia's centre (25°56′49.3″S, 133°12′34.7″E) for the location
    zoom_start=4,
    tiles="CartoDB DarkMatter",  # a dark basemap to make heatmap stand out
)

# 2. Add State/Territory Polygons
folium.GeoJson(
    gdf_territories,
    tooltip=folium.GeoJsonTooltip(
        fields=["STATE_NAME_2021"],
        aliases=["State/Territory:"]
    ),
    style_function=lambda feature:{
        "fillColor": "dark grey", # somehow dark grey does not work
        "color": "white",
        "weight": 0.5,
    }
).add_to(aus_map)

# 3. Create heatmap with Fires GDF that shows intensity of different fires (frp)
HeatMapWithTime(
    data,
    index=time_index,
    radius=8,
    auto_play=True,
    max_opacity=0.8
).add_to(aus_map)

# save the map (display does not work bc data file is too big)
aus_map.save("animated_heatmap_MODIS.html")